Train,      Test  (OWT 4096 blocks step by 2048)
3,757,207, 417,680

# QNA: Train,   Test
## 512: 2_061_221, 229_528
## 256: 2_414_072, 268_024
## 128: 3_041_299, 337_519

In [1]:
import numpy as np

In [2]:
folder = '/home/user/Desktop/ml/gpt_dt/datasets/fineweb_t2/'
splits = ['s7', 'val']#['s0', 's1', 's2', 's3', 's4', 's5', 's6', 's7']
abr = "fw"
block = 4096
step = 4096

In [3]:
def generate_shuffle_map(data_path, output_path, block_size, map_step_size, offset=0, seed=42):

    data = np.memmap(data_path, dtype=np.uint16, mode='r')
    total_tokens = len(data)
    num_chunks = ((total_tokens - 1 - block_size) // map_step_size) + 1
    print(f"Generating map for {num_chunks} chunks...")

    m = np.memmap(output_path, dtype=np.int64, mode='w+', shape=(int(num_chunks),))

    m[:] = np.arange(num_chunks) + offset # Fill  with sequential numbers
    
    rng = np.random.default_rng(seed)
    rng.shuffle(m)
    
    m.flush() # Flush to disk and close
    print(f"Shuffle map saved to {output_path}")
    
    return num_chunks

for split in splits:
    generate_shuffle_map(f'{folder}shards/{split}.bin', f'{folder}{block}b_{step}s_maps/{split}_map0.bin', block, step, offset=0)

Generating map for 1865254 chunks...
Shuffle map saved to /home/user/Desktop/ml/gpt_dt/datasets/fineweb_t2/4096b_4096s_maps/s7_map0.bin
Generating map for 2441 chunks...
Shuffle map saved to /home/user/Desktop/ml/gpt_dt/datasets/fineweb_t2/4096b_4096s_maps/val_map0.bin


In [ ]:
# shuffle some extra indices into only the end of the shuffle
# this works if the math chunks are appended right on to the end of the text data in the data bin file
filename = f'{folder}/all_train_map0-1024block-512step.bin'
dtype = np.int64 

original_num_chunks = 15_616_533
new_indices_count = 650_000

new_math_indices = np.arange(new_indices_count, dtype=dtype) + original_num_chunks

with open(filename, 'ab') as f:
    f.write(new_math_indices.tobytes())
    f.flush()

total_chunks = original_num_chunks + new_indices_count
m = np.memmap(filename, dtype=dtype, mode='r+', shape=(total_chunks,))

print(f"Total chunks in map: {len(m)} - {total_chunks}") # Should be ~16.2M

start_shuffle_at = total_chunks - (2_500_000 + new_indices_count)
end_shuffle_at = total_chunks

tail_indices = np.copy(m[start_shuffle_at:end_shuffle_at])
np.random.shuffle(tail_indices)

m[start_shuffle_at:end_shuffle_at] = tail_indices

m.flush()
print("Map fixed. First Math sample should now be > 13,000,000.")

Total chunks in map: 16266533 - 16266533
Map fixed. First Math sample should now be > 13,000,000.


In [ ]:
# Load the map (automatically detects size)
m = np.memmap(f'{folder}/all_train_map0-1024block-512step.bin', dtype=np.int64, mode='r')
threshold = 15_616_533
print(f"Total chunks in map: {len(m)}")

start_leak = np.any(m[:100_000] >= threshold)
print(f"Math found in first 10k samples? {start_leak}") # Should be False

mid_leak = np.any(m[10_000_000:10_100_000] >= threshold)
print(f"Math found at sample 10M? {mid_leak}") # Should be False

first_math_idx = np.where(m >= threshold)[0][0]
print(f"First Math sample appears at index: {first_math_idx}")
# Based on your logic, this should be around 13,116,533
m[:10]

Total chunks in map: 16266533
Math found in first 10k samples? False
Math found at sample 10M? False
First Math sample appears at index: 13116533


memmap([ 9196937, 13158514, 10793101,  8617859,  3855890, 15289092,
         5905814, 14248504, 15117240,  8658685])